# 🏆 DS102 A100 Championship Pipeline — Chinh phục Top 1 (`> 0.80819`)
Notebook này được thiết kế chuyên biệt cho **NVIDIA A100-SXM4-80GB** trên Google Colab:
1. **An toàn dữ liệu tuyệt đối (Google Drive Live Sync):** Gắn kết trực tiếp Google Drive (`/content/drive/MyDrive/DS102_Colab_Cache`). Ngay khi **từng Fold** và **từng mô hình** chạy xong, xác suất OOF/Dev/Test (`.npy`) được sao lưu tức thì lên Drive — dù tắt trình duyệt hay đứt kết nối giữa chừng cũng **không bao giờ mất kết quả** và tự động chạy tiếp (resume) từ đúng Fold đang dở!
2. **4 Kiến trúc Transformer Hạng nặng (`BF16 Tensor Cores`):**
   - `xlm_ctx2`: **5-Fold `xlm-roberta-base`** (Variant B Product-First, `MAX_LEN=288`)
   - `xlm_large_ctx`: **5-Fold `xlm-roberta-large`** (560M tham số, 24 tầng, `MAX_LEN=384`, `[CLS]+Mean+Max` + Layerwise LR Decay)
   - `mdeberta_ctx`: **5-Fold `microsoft/mdeberta-v3-base`** (Disentangled Relative Attention đa ngôn ngữ, `MAX_LEN=384`)
   - `phobert_large_ctx`: **5-Fold `vinai/phobert-large`** (370M tham số, 24 tầng Tiếng Việt chuyên sâu, `MAX_LEN=254`)
3. **Siêu Tổ hợp 6 Họ Mô hình Tầng 2 (`build_submission_17.py`) & Tự động nộp Kaggle CLI:** Kết hợp toàn bộ 10 luồng xác suất Tầng 1 + đặc trưng nhận diện sàn `Tiki/Shopee` + `Bagged XGBoost + Bagged LightGBM + ExtraTrees + CatBoost + Residual Meta-MLP`, tối ưu trọng số Nested-CV và tự động nộp `submission_17.csv` lên Kaggle!

In [ ]:
# 1. Kiểm tra GPU (Khuyên dùng Runtime > Change runtime type > A100 GPU)
!nvidia-smi

In [ ]:
# 2. Gắn kết Google Drive để sao lưu tự động 100% kết quả huấn luyện (chống mất file khi tắt runtime)
import os
from google.colab import drive

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/DS102_Colab_Cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"✅ Đã kết nối thư mục lưu trữ bền vững trên Google Drive: {CACHE_DIR}")
print("   Hiện có các file đã lưu:", sorted(os.listdir(CACHE_DIR)))

In [ ]:
# 3. Tải mã nguồn mới nhất từ GitHub & cài đặt đầy đủ thư viện (bao gồm sentencepiece/tiktoken cho mDeBERTa-v3)
import os
%cd /content
if not os.path.exists("/content/KaggleComp"):
    !git clone https://github.com/24520559-Nh0c3c/KaggleComp.git
%cd /content/KaggleComp
!git fetch origin main && git reset --hard origin/main
!pip install -q -U transformers sentencepiece tiktoken protobuf lightgbm xgboost catboost kaggle
print("✅ Đã đồng bộ code mới nhất và cài đặt xong thư viện!")

In [ ]:
# 4. Cấu hình Kaggle API (Tự động lưu vào Google Drive để các lần chạy sau không cần tải lên lại!)
import os, shutil
from google.colab import files

CACHE_DIR = "/content/drive/MyDrive/DS102_Colab_Cache"
DRIVE_KAGGLE = os.path.join(CACHE_DIR, "kaggle.json")
LOCAL_KAGGLE = "/root/.kaggle/kaggle.json"
os.makedirs("/root/.kaggle", exist_ok=True)

if os.path.exists(DRIVE_KAGGLE):
    shutil.copy2(DRIVE_KAGGLE, LOCAL_KAGGLE)
    os.chmod(LOCAL_KAGGLE, 0o600)
    print("✅ Đã tự động khôi phục kaggle.json từ Google Drive! Không cần upload lại.")
elif not os.path.exists(LOCAL_KAGGLE):
    print("👉 Lần đầu tiên: Hãy chọn file kaggle.json của bạn (sẽ được tự động lưu vào Drive cho các lần sau):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".json"):
            shutil.copy2(fn, LOCAL_KAGGLE)
            os.chmod(LOCAL_KAGGLE, 0o600)
            if os.path.exists(CACHE_DIR):
                shutil.copy2(fn, DRIVE_KAGGLE)
            print("✅ Cấu hình Kaggle API thành công và đã sao lưu vào Google Drive!")
            break
else:
    print("✅ Kaggle API đã sẵn sàng!")

In [ ]:
# 5. Huấn luyện bộ 4 Foundation Models trên A100 (BF16 + lưu dự phòng từng Fold lên Google Drive)
# Tổng thời gian dự kiến trên A100: ~35-40 phút cho toàn bộ 20 mô hình (4 kiến trúc x 5 Fold).
# Nếu bị ngắt giữa chừng, chạy lại ô này sẽ tự động bỏ qua các mô hình/Fold đã lưu trong Drive!
!python -u train_a100_championship_models.py --cache-dir /content/drive/MyDrive/DS102_Colab_Cache

In [ ]:
# 6. Huấn luyện Siêu Tổ hợp Stacking Tầng 2 (6 họ mô hình trên 15,896 mẫu) + Sao lưu Drive + Tự động nộp Kaggle!
!python -u build_submission_17.py --cache-dir /content/drive/MyDrive/DS102_Colab_Cache

In [ ]:
# 7. Kiểm tra toàn bộ file đã sao lưu an toàn trên Google Drive & tải submission_17.csv về máy
import os
from google.colab import files

CACHE_DIR = "/content/drive/MyDrive/DS102_Colab_Cache"
print("📦 Danh sách các file đã được bảo toàn vĩnh viễn trong Google Drive của bạn:")
for fn in sorted(os.listdir(CACHE_DIR)):
    fp = os.path.join(CACHE_DIR, fn)
    if os.path.isfile(fp) and fn != "kaggle.json":
        print(f"   - {fn:36s} ({os.path.getsize(fp)/1024:.1f} KB)")

if os.path.exists("submission_17.csv"):
    files.download("submission_17.csv")
    print("\n✅ Đã tải thêm bản sao submission_17.csv về máy tính!")